# My Tiny Torch playground

Install this project with `python -m pip install -e '.[dev,notebook]'` and select its environment as your notebook kernel. Run the cells from top to bottom, then edit the inputs.

The public Tensor API is wired to operation classes. This notebook also calls `Operation.apply(...)` directly to show the execution boundary. Every operation runs immediately on CPU float32 storage. Gradient history is recorded when requested; backward calculations are not implemented.

The inspector accesses private `_storage` and `_ctx` for learning about implementation details. Application code should use public methods.

In [ ]:
import numpy as np

from tinytorch import Tensor
from tinytorch.ops import (
    Add, Sub, Mul, Div, MatMul, Stack, GetItem, Transpose,
    Permute, Reshape, View, Expand, Clone, Contiguous, Sum, Mean, Max,
)

# Use the project's virtual environment as this notebook's kernel.
# Installation instructions are in README.md.
# After editing library files, restart the kernel and run the cells again.

In [ ]:
def show(name, tensor):
    array = tensor._storage
    ctx = tensor._ctx
    print(f"{name}:\n{tensor.numpy()}")
    print(f"  shape={tensor.shape}, ndim={tensor.ndim}, dtype={tensor.dtype}")
    print(f"  strides(bytes)={tensor.strides}, strides(elements)="
          f"{tuple(s // array.itemsize for s in tensor.strides)}")
    print(f"  contiguous={tensor.is_contiguous()}, numel={tensor.numel()}, "
          f"logical bytes={array.nbytes}")
    print(f"  requires_grad={tensor.requires_grad}, "
          f"operation={type(ctx).__name__ if ctx is not None else None}")
    print()

def same_storage(a, b):
    shared = np.shares_memory(a._storage, b._storage)
    print("Same Tensor object:", a is b)
    print("Same ndarray object:", a._storage is b._storage)
    print("Overlapping element storage:", shared)
    return shared

# shares_memory checks overlapping bytes, not common ancestry.
# Empty arrays have no element bytes to overlap.
def expect_error(action):
    try:
        action()
    except Exception as error:
        print(f"{type(error).__name__}: {error}")
    else:
        raise AssertionError("Expected an exception, but the call succeeded.")

## Construction, scalars, and empty tensors

Public construction copies into independent row-major float32 storage. A scalar has shape `()` but is still backed by an ndarray. Boolean inputs become `1.0` and `0.0`.

In [ ]:
x = Tensor([[1, 2, 3], [4, 5, 6]])
show("x", x)
show("scalar", Tensor(2))
show("empty", Tensor([]))
show("boolean input", Tensor([True, False]))

In [ ]:
source = np.arange(6, dtype=np.float32).reshape(2, 3)
constructed = Tensor(source)
print("Constructor shares source storage:", np.shares_memory(source, constructed._storage))
source[0, 0] = 99
show("constructed (unchanged)", constructed)

copied = Tensor(x)
assert not same_storage(x, copied)
show("numeric strings converted", Tensor(["1", "2"]))
expect_error(lambda: Tensor([1, 2], dtype="float64"))

## Public Tensor API

These are the normal user-facing calls; no direct operation imports are needed.

In [ ]:
public_x = Tensor([[1, 2, 3], [4, 5, 6]])
show("public addition", public_x + Tensor([10, 20, 30]))
show("public transpose", public_x.transpose(0, 1))
show("public flat view", public_x.view(6))
show("public row sums", public_x.sum(axis=1, keepdims=True))
show("public stack", Tensor.stack([public_x, public_x]))

## Arithmetic and broadcasting

Scalars pass straight to NumPy. Reverse subtraction and division keep operand order by placing the scalar first in `apply`. These direct calls bypass the current Tensor arithmetic methods.

In [ ]:
show("x + 2", Add.apply(x, 2))
show("10 - x", Sub.apply(10, x))
show("x * 2", Mul.apply(x, 2))
show("12 / x", Div.apply(12, x))
show("scalar Tensor + scalar", Add.apply(Tensor(3), np.float32(2)))
assert not same_storage(x, Add.apply(x, 2))

In [ ]:
bias = Tensor([10, 20, 30])
show("bias", bias)
show("x + bias", Add.apply(x, bias))
show("column + row", Add.apply(Tensor([[10], [20]]), bias))
expect_error(lambda: Add.apply(x, Tensor([1, 2])))
expect_error(lambda: Add.apply(x, "2"))

# NumPy may promote the result. Wrapping currently rejects non-float32 output.
expect_error(lambda: Add.apply(x, np.float64(1e100)))

## Matrix multiplication

A matrix times a matrix produces a matrix. A vector dot product produces a zero-dimensional Tensor.

In [ ]:
weights = Tensor([[1, 0], [0, 1], [1, 1]])
show("x @ weights", MatMul.apply(x, weights))
show("dot product", MatMul.apply(Tensor([1, 2, 3]), Tensor([4, 5, 6])))

## Transpose and permute

The element buffer stays in place. NumPy swaps the shape and stride metadata.

In [ ]:
xt = Transpose.apply(x, dim0=0, dim1=1)
show("x", x)
show("xt", xt)
assert same_storage(x, xt)

cube = Tensor(np.arange(24, dtype=np.float32).reshape(2, 3, 4))
show("permuted cube", Permute.apply(cube, axes=(2, 0, 1)))

## Indexing: views and copies

`key` is the index passed to NumPy. Basic slicing shares storage, advanced indexing allocates, and selecting one element returns a scalar-shaped Tensor.

In [ ]:
sliced = GetItem.apply(x, key=(slice(1, None), slice(None, 2)))
show("x[1:, :2]", sliced)
assert same_storage(x, sliced)

selected = GetItem.apply(x, key=[1, 0])
show("x[[1, 0]]", selected)
assert not same_storage(x, selected)
show("x[1, 2]", GetItem.apply(x, key=(1, 2)))
show("x[:, ::-1]", GetItem.apply(x, key=(slice(None), slice(None, None, -1))))

## Reshape versus strict view

Reshape may copy. View must reuse storage or raise. Reshaping to the existing shape doesn't require packing a transposed array.

In [ ]:
flat = Reshape.apply(x, shape=(6,))
show("flat x", flat)
assert same_storage(x, flat)

same_shape = Reshape.apply(xt, shape=xt.shape)
assert same_storage(xt, same_shape)

flat_transpose = Reshape.apply(xt, shape=(6,))
show("flattened transpose", flat_transpose)
assert not same_storage(xt, flat_transpose)
expect_error(lambda: View.apply(xt, shape=(6,)))
show("strict view of x", View.apply(x, shape=(3, 2)))

In [ ]:
# A noncontiguous array can still support some no-copy reshapes.
base = Tensor(np.arange(24, dtype=np.float32).reshape(4, 6))
stepped = GetItem.apply(base, key=(slice(None), slice(None, None, 2)))
stepped_flat = View.apply(stepped, shape=(-1,))
show("stepped", stepped)
show("stepped_flat", stepped_flat)
assert same_storage(stepped, stepped_flat)
show("empty view", View.apply(Tensor([]), shape=(0, 2)))

## Expand and zero strides

Expand reuses input values with stride zero. Its ndarray is read-only. Addition consumes this view and allocates its own result.

In [ ]:
expanded = Expand.apply(bias, shape=(2, 3))
show("expanded bias", expanded)
assert same_storage(bias, expanded)
print("Backing array writable:", expanded._storage.flags.writeable)
show("x + expanded", Add.apply(x, expanded))
expect_error(lambda: Expand.apply(bias, shape=(2, 4)))

## Clone, contiguous, and NumPy export

Clone always copies element storage. Contiguous packs a noncontiguous array. A future `Tensor.contiguous()` wrapper should return `self` when already contiguous; calling the operation directly still creates a new Tensor wrapper.

`numpy()` exports a copy. Its strides need not match the Tensor's actual strides.

In [ ]:
cloned = Clone.apply(xt)
packed = Contiguous.apply(xt)
show("clone of transpose", cloned)
show("packed transpose", packed)
assert not same_storage(xt, cloned)
assert not same_storage(xt, packed)

already_contiguous = Contiguous.apply(x)
assert same_storage(x, already_contiguous)

exported = xt.numpy()
print("Actual Tensor strides:", xt.strides)
print("Exported copy strides:", exported.strides)
print("Export overlaps Tensor:", np.shares_memory(exported, xt._storage))
exported[:] = -100
show("xt after editing exported copy", xt)

## Reductions and keepdims

An axis names the dimension to combine. Keeping it at length one makes broadcasting back easier.

In [ ]:
show("sum all", Sum.apply(x))
show("sum rows (axis=0)", Sum.apply(x, axis=0))
show("sum columns (axis=1)", Sum.apply(x, axis=1))
row_mean = Mean.apply(x, axis=1, keepdims=True)
show("row mean, keepdims=True", row_mean)
show("centered rows", Sub.apply(x, row_mean))
show("max per row", Max.apply(x, axis=1))
show("sum multiple axes", Sum.apply(cube, axis=(0, 2), keepdims=True))

## Stack and construction history

Both stacking paths allocate output storage. Constructor stacking makes a fresh leaf. `Stack.apply` records the operation when an input requires gradients. It does not calculate gradients yet.

In [ ]:
a = Tensor([1, 2], requires_grad=True)
b = Tensor([3, 4], requires_grad=True)
constructed_stack = Tensor([a, b])
operation_stack = Stack.apply(a, b, axis=0)
show("constructor stack", constructed_stack)
show("operation stack", operation_stack)
show("stack axis=1", Stack.apply(a, b, axis=1))
assert constructed_stack._ctx is None
assert operation_stack._ctx.inputs[0] is a
assert operation_stack._ctx.inputs[1] is b
assert not same_storage(a, operation_stack)

## Inspect the operation graph

Each `apply` creates a separate operation object. Connections reference input Tensors, not copies of their values. This is eager execution: values have already been computed. These connections are for future backward traversal.

In [ ]:
leaf = Tensor([1, 2, 3], requires_grad=True)
scaled = Mul.apply(leaf, 2)
shifted = Add.apply(scaled, 1)
loss = Sum.apply(shifted)
show("loss", loss)
print("Loss operation:", type(loss._ctx).__name__)
print("Loss input is shifted:", loss._ctx.inputs[0] is shifted)
print("Shifted input is scaled:", shifted._ctx.inputs[0] is scaled)
print("Scaled inputs:", type(scaled._ctx.inputs[0]).__name__, scaled._ctx.inputs[1])
other_scaled = Mul.apply(leaf, 2)
print("Separate operation objects:", scaled._ctx is not other_scaled._ctx)
expect_error(lambda: loss._ctx.backward(np.asarray(1, dtype=np.float32)))

## Your experiments

Try another shape, a negative axis, different slices, or chained operations. Predict the values, shape, strides, and storage sharing before running.

In [ ]:
# Edit this cell.
experiment = Transpose.apply(x, dim0=0, dim1=1)
show("experiment", experiment)
same_storage(x, experiment)